In [ ]:
!pip install transformers datasets sentencepiece accelerate -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

In [ ]:
BASE_MODEL   = "csebuetnlp/mT5_multilingual_XLSum"
LANG_TOKEN   = "<2bn>"          # Bengali language token — mandatory for this model
DATA_CSV     = "resume_summarization_dataset.csv"
OUTPUT_DIR   = "bangla_resume_summarizerMT5"

MAX_INPUT    = 512
MAX_TARGET   = 128
BATCH_SIZE   = 4
EPOCHS       = 5
LR           = 5e-4

In [ ]:
df = pd.read_csv(DATA_CSV).dropna(subset=["Input_Text", "Reference_Summary"])
df = df[["Input_Text", "Reference_Summary"]].reset_index(drop=True)

print(f"Total pairs: {len(df)}")

# 80/20 train/validation split
split     = int(len(df) * 0.8)
train_df  = df.iloc[:split].reset_index(drop=True)
val_df    = df.iloc[split:].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

Total pairs: 396
Train: 316  |  Val: 80


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)


def preprocess(batch):
    # Prepend Bengali language token to every input
    inputs  = [LANG_TOKEN + " " + t.strip() for t in batch["Input_Text"]]
    targets = [s.strip() for s in batch["Reference_Summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
    text_target=targets,
    max_length=MAX_TARGET,
    truncation=True,
    padding=False,
)

    # Replace padding token id with -100 (ignored in loss computation)
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_ds.map(preprocess, batched=True,
                                remove_columns=["Input_Text", "Reference_Summary"])
tokenized_val   = val_ds.map(preprocess,   batched=True,
                                remove_columns=["Input_Text", "Reference_Summary"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
print(f"Model parameters: {model.num_parameters():,}")

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model parameters: 582,401,280


In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=50,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_total_limit=2,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

In [ ]:
trainer = Seq2SeqTrainer(
    model         = model,
    args          = args,
    train_dataset = tokenized_train,
    eval_dataset  = tokenized_val,
    processing_class=tokenizer,
    data_collator = data_collator,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting fine-tuning...")
trainer.train()

Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.968941,1.112221
2,0.856486,0.954938
3,0.598892,0.890829
4,0.496979,0.887174
5,0.473462,0.886270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=395, training_loss=0.821700572363938, metrics={'train_runtime': 1194.0029, 'train_samples_per_second': 1.323, 'train_steps_per_second': 0.331, 'total_flos': 709985190739968.0, 'train_loss': 0.821700572363938, 'epoch': 5.0})

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nFine-tuned model saved to '{OUTPUT_DIR}/'")
print("Download this folder and place it next to step3_abstractive_summarize_cv.py")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved to 'bangla_resume_summarizer/'
Download this folder and place it next to step3_abstractive_summarize_cv.py


In [ ]:
# ── Save to Google Drive (skip checkpoint folders only) ───────────────────────
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/bangla_resume_summarizer"

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

for item in os.listdir(OUTPUT_DIR):
    if item.startswith("checkpoint-"):
        print(f"  Skipped: {item}")
        continue

    src  = os.path.join(OUTPUT_DIR, item)
    dest = os.path.join(DRIVE_SAVE_PATH, item)

    if os.path.isdir(src):
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)

    print(f"  Copied: {item}")

print(f"\nDone. Model saved to Google Drive: '{DRIVE_SAVE_PATH}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Copied: model.safetensors
  Copied: tokenizer_config.json
  Copied: config.json
  Skipped: checkpoint-316
  Copied: tokenizer.json
  Skipped: checkpoint-395
  Copied: generation_config.json
  Copied: training_args.bin

Done. Model saved to Google Drive: '/content/drive/MyDrive/bangla_resume_summarizer'


In [ ]:
print("\n--- Sanity Check (first 3 val samples) ---")

device = model.device

for i in range(min(3, len(val_df))):

    input_text = LANG_TOKEN + " " + val_df.iloc[i]["Input_Text"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT
    )

    # Move tensors to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    pred = tokenizer.decode(
        out[0],
        skip_special_tokens=True
    )

    print(f"\nInput    : {val_df.iloc[i]['Input_Text'][:80]}...")
    print(f"Reference: {val_df.iloc[i]['Reference_Summary']}")
    print(f"Generated: {pred}")


--- Sanity Check (first 3 val samples) ---

Input    : স্বাস্থ্যসেবা খাতে ডেটা অ্যানালিস্ট হিসেবে কাজ করা, যেখানে ক্লিনিকাল ডেটা বিশ্লে...
Reference: স্বাস্থ্যসেবা ক্ষেত্রে ডেটা অ্যানালিস্ট হিসেবে কাজ করতে ইচ্ছুক। ক্লিনিকাল ডেটা বিশ্লেষণের মাধ্যমে রোগীর সেবার মান উন্নত করা এবং হাসপাতালের কার্যকারিতা বৃদ্ধি করা লক্ষ্য।
Generated: ডেটা অ্যানালিস্ট হিসেবে স্বাস্থ্যসেবা খাতে কাজ করতে ইচ্ছুক। ক্লিনিকাল বিশ্লেষণের মাধ্যমে রোগীর সেবার মান ও কার্যকারিতা বৃদ্ধি করা সম্ভব।

Input    : ক্লিনিকাল ডেটা অ্যানালিস্ট । সেপ্টেম্বর ২০২১ - বর্তমান । রোগীর তথ্য এবং ক্লিনিকা...
Reference: সেপ্টেম্বর ২০২১ থেকে বর্তমান পর্যন্ত ক্লিনিকাল ডেটা অ্যানালিস্ট হিসেবে রোগীর তথ্য ও ক্লিনিকাল ট্রায়াল ডেটা বিশ্লেষণ করেন। ডেটা প্রাইভেসি ও কমপ্লায়েন্স বজায় রেখে হাসপাতালের রিসোর্স অ্যালোকেশন রিপোর্ট প্রস্তুত করেন।
Generated: সেপ্টেম্বর ২০২১ থেকে বর্তমান পর্যন্ত তিনি ক্লিনিকাল ডেটা অ্যানালিস্ট হিসেবে কাজ করছেন, যেখানে রোগীর তথ্য বিশ্লেষণ করে প্রাইভেসি ও কমপ্লায়েন্স নিশ্চিত করেছেন। এছাড়া, হাসপাতালের রিসোর্স অ্যালোকেশন